# 5-2 Token単位
Janomeで日本語をTokenに分け、同じ10件を分類します。


In [ ]:
training_data = [('沖縄の海でシュノーケリングを楽しみたい', 1), ('京都まで新幹線で行き寺を見学する', 1), ('ホテルを予約して北海道を三日間観光する', 1), ('成田からパリへの航空券を探している', 1), ('温泉旅館に泊まり箱根を散策したい', 1), ('Pythonでファイルを読み込む方法を調べる', 0), ('今日の夕食はカレーを作る予定だ', 0), ('会社の会議資料を明日までに作成する', 0), ('野球の試合結果をニュースで確認した', 0), ('新しいパソコンのメモリを増設したい', 0)]
inference_data = ['来月、飛行機で沖縄へ行く', '京都の写真をパソコンに保存する', '駅まで歩いて会社へ行く']


In [ ]:
!pip -q install janome
from janome.tokenizer import Tokenizer
import math
t=Tokenizer()
def split_tokens(text):
    return [x.surface for x in t.tokenize(text) if x.part_of_speech.split(",")[0]!="記号"]

def train_model(data, split):
    class_n=[0,0]; counts=[{},{}]; totals=[0,0]; vocab=set()
    for text,y in data:
        class_n[y]+=1
        for x in split(text):
            vocab.add(x); counts[y][x]=counts[y].get(x,0)+1; totals[y]+=1
    return class_n,counts,totals,vocab

def predict(text, model, split):
    class_n,counts,totals,vocab=model; scores=[]
    for y in [0,1]:
        s=math.log(class_n[y]/sum(class_n))
        for x in split(text):
            s+=math.log((counts[y].get(x,0)+1)/(totals[y]+len(vocab)))
        scores.append(s)
    return int(scores[1]>scores[0]),scores

model=train_model(training_data,split_tokens)
for text in inference_data:
    ans,s=predict(text,model,split_tokens)
    print(text)
    print("Token =", " / ".join(split_tokens(text)))
    print("判定 =", "旅行" if ans else "旅行以外", [round(v,2) for v in s], "\n")
